In [ ]:
from src.sentence_representations import SentenceRepresentations
from src.word_representations import WordRepresentations
from src.spd_matrix_learner import SPDMatrixLearnerCfg
from src.stimulis import Stimulis
from src.trainer import Trainer
from src.feature_importance import FeatureImportance
import pandas as pd
import torch
from tqdm.auto import tqdm
from pydantic import BaseModel, ConfigDict, Field, PrivateAttr
from loguru import logger
import sys
import plotly.express as px
from exca import TaskInfra
import typing as tp
import numpy as np
from copy import deepcopy

logger.remove()
logger.add(sys.stderr, level="WARNING")

In [ ]:
importances = []
spearman = []
run_id = logger.add("run.log", level="WARNING")
for distance in [2, "cosine"]:
    for token_aggregation in ["mean", "first"]:
        for layer in range(13):
            fi = FeatureImportance(
                trainer={
                    "model": {},
                    "dataframe": {"csv_path": "datasets/relative_clause.csv"},
                    "representations": {
                        "token_aggregation": token_aggregation,
                        "layer": layer,
                    },
                    "dataset": {"distance": distance},
                }
            )
            folder = fi.infra.uid_folder()
            folder.mkdir(parents=True, exist_ok=True)
            with open(folder / "info.log", "w") as logs:
                logger_id = logger.add(logs, level="INFO")
                i, s = fi.compute()
                logger.remove(logger_id)
                for k, v in {
                    "distance": distance,
                    "token_aggregation": token_aggregation,
                    "layer": layer,
                }.items():
                    i[k] = v
                    s[k] = v
                importances.append(i)
                spearman.append(s)
logger.remove(run_id)
importances = pd.concat(importances)
spearman = pd.DataFrame(spearman)

In [ ]:
fig = px.line(
    importances.query("mean > 0.05"),
    x="layer",
    y="mean",
    color="Feature",
    error_y="std",
    facet_col="distance",
    facet_row="token_aggregation",
    title="Feature Importance",
)
fig.write_html("feature_importance.html")

In [ ]:
spearman["setup"] = (
    spearman.distance.replace(2, "euclidean").astype(str)
    + " "
    + spearman.token_aggregation
)

In [ ]:
fig = px.line(
    spearman,
    x="layer",
    y="mean",
    error_y="std",
    color="setup",
    title="Feature Importance",
)
fig.write_html("spearman.html")

In [ ]:
importances = []
spearman = []
run_id = logger.add("run.log", level="WARNING")
for model_name in ["gpt2", "bert-base-uncased"]:
    for param in ["none", "sym", "diagonal", "exp", "cholesky"]:
        for layer in range(13):
            logger.warning(f"param: {param}, layer: {layer}")
            fi = FeatureImportance(
                trainer={
                    "model": {"param": param},
                    "dataframe": {"csv_path": "datasets/relative_clause.csv"},
                    "representations": {
                        "layer": layer,
                        "model_name": model_name,
                    },
                    "dataset": {},
                }
            )
            folder = fi.infra.uid_folder()
            folder.mkdir(parents=True, exist_ok=True)
            with open(folder / "info.log", "w") as logs:
                logger_id = logger.add(logs, level="INFO")
                i, s = fi.compute()
                logger.remove(logger_id)
                for k, v in {
                    "param": param,
                    "layer": layer,
                }.items():
                    i[k] = v
                    s[k] = v
                importances.append(i)
                spearman.append(s)
logger.remove(run_id)
importances = pd.concat(importances)
spearman = pd.DataFrame(spearman)

In [ ]:
importances = []
spearman = []
run_id = logger.add("run.log", level="WARNING")
for model_name in ["bert-base-uncased", "gpt2"]:
    for param in ["none", "sym", "diagonal", "exp", "cholesky"]:
        for layer in range(13):
            logger.warning(f"param: {param}, layer: {layer}")
            fi = FeatureImportance(
                trainer={
                    "model": {"param": param},
                    "dataframe": {"csv_path": "datasets/svo_word_level.csv"},
                    "representations": {
                        "level": "word",
                        "layer": layer,
                        "model_name": model_name,
                    },
                    "dataset": {},
                }
            )
            folder = fi.infra.uid_folder()
            folder.mkdir(parents=True, exist_ok=True)
            with open(folder / "info.log", "w") as logs:
                logger_id = logger.add(logs, level="INFO")
                i, s = fi.compute()
                logger.remove(logger_id)
                for k, v in {
                    "model": model_name,
                    "param": param,
                    "layer": layer,
                }.items():
                    i[k] = v
                    s[k] = v
                importances.append(i)
                spearman.append(s)
logger.remove(run_id)
importances = pd.concat(importances)
spearman = pd.DataFrame(spearman)

In [ ]:
fig = px.line(
    importances.query("mean > 0.1"),
    x="layer",
    y="mean",
    color="Feature",
    error_y="std",
    facet_col="model",
    facet_row="param",
)
fig.write_html(".figs/fi_param_word_level.html")
fig

In [ ]:
ordered = None
for param in df:
    tmp = df.sort_values(
        ["layer", param], ascending=[True, False]
    ).reset_index()
    tmp.loc[tmp[param] < 0.1, "Feature"] = np.nan
    if ordered is None:
        tmp = tmp[["layer", "Feature"]].rename(columns={"Feature": param})
        ordered = tmp
    else:
        ordered[param] = tmp["Feature"]

In [ ]:
ordered

In [ ]:
cholesky

In [ ]:
df.groupby("layer").rank()

In [ ]:
df = importances[importances.model == "bert-base-uncased"]
# df.loc[df["mean"] < 0.1, "mean"] = np.nan
df = df.pivot(
    index=["layer", "Feature"],
    columns=["param"],
    values="mean",
)
df

In [ ]:
px.imshow(
    df.corr("spearman"),
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
)

In [ ]:
fig = px.line(
    spearman,
    x="layer",
    y="mean",
    color="param",
    error_y="std",
    facet_col="model",
    title="Spearman",
)
fig.write_html(".figs/spearman_param_word_level.html")
fig

In [ ]:
importances = []
spearman = []
run_id = logger.add("run.log", level="WARNING")
for layer in tqdm(range(13)):
    fi = FeatureImportance(
        trainer={
            "dataframe": {"csv_path": "datasets/svo_word_level.csv"},
            "representations": {
                "layer": layer / 13,
                "level": "word",
                # "model_name": "distilbert/distilroberta-base",
                # "model_name": "bert-base-uncased",
                "model_name": "gpt2",
            },
        }
    )
    folder = fi.infra.uid_folder()
    folder.mkdir(parents=True, exist_ok=True)
    with open(folder / "info.log", "w") as logs:
        logger_id = logger.add(logs, level="INFO")
        i, s = fi.compute()
        logger.remove(logger_id)
        i["layer"] = layer
        s["layer"] = layer
        importances.append(i)
        spearman.append(s)
logger.remove(run_id)
importances = pd.concat(importances)
spearman = pd.DataFrame(spearman)

In [ ]:
fig = px.line(
    spearman,
    x="layer",
    y="mean",
    error_y="std",
    title="Spearman",
)
# fig.write_html("figs/spearman_param.html")
fig

In [ ]:
fig = px.line(
    importances.query("mean > 0.05"),
    x="layer",
    y="mean",
    color="Feature",
    error_y="std",
)
# fig.write_html("figs/fi_param.html")
fig